In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
OS_Scan_data_total  = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/xNIDS/Data/OS_Scan_dataset.csv")
OS_Scan_data_total

In [ ]:
OS_Scan_label_total  = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/xNIDS/Data/OS_Scan_labels.csv")
OS_Scan_label_total

In [ ]:
# Drop the last row
OS_Scan_label_total = OS_Scan_label_total.drop(OS_Scan_label_total.index[-1])
OS_Scan_label_total

In [ ]:
# Add labels as a new column to the train dataset
OS_Scan_data_total['labels'] = OS_Scan_label_total.iloc[:, 1]

# Save the updated train dataset to a new CSV file
#OS_Scan_data_total.to_csv('/content/drive/MyDrive/Colab Notebooks/xNIDS/Data/merged_OS_Scan.csv', index=False)


In [ ]:
import psutil

# Get disk usage information
disk_usage = psutil.disk_usage('/')

# Print the used disk space in bytes
print("Used disk space:", disk_usage.used)

# Print the used disk space in a human-readable format
print("Used disk space:", psutil.disk_usage('/').used / (1024**3), "GB")


In [ ]:
# Calculate the distribution of each label
# Get the label column (assuming it is the last column)
label_column = OS_Scan_data_total.iloc[:, -1]
# Calculate the distribution of each label
label_distribution = label_column.value_counts(normalize=True)

# Print the label distribution
print(label_distribution)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split

# Assuming you have your data loaded into the data_array
# Remove the first column from the data
feature_data = OS_Scan_data_total.iloc[:, :-1]
label_data = OS_Scan_data_total.iloc[:, -1]
#del OS_Scan_data_total
#del OS_Scan_label_total


# Calculate the index for splitting the data
split_index = 1000000 #1M for training



from sklearn.preprocessing import RobustScaler

# Create a RobustScaler object
scaler = RobustScaler()

# Normalize each column using RobustScaler
normalized_data = scaler.fit_transform(feature_data)

# Create a new DataFrame with the normalized values
df = pd.DataFrame(normalized_data, columns=feature_data.columns)


from sklearn.preprocessing import MinMaxScaler

# Assuming your DataFrame is called 'df'

# Create an instance of MinMaxScaler
scaler = MinMaxScaler()

# Fit the scaler on the entire DataFrame
scaler.fit(df)

# Transform the entire DataFrame with the scaler
normalized_df = pd.DataFrame(scaler.transform(df), columns=df.columns)



# Split the data into training and testing sets
X_train =  normalized_df[:split_index]
X_test =  normalized_df[split_index:]
Y_test = label_data[split_index:]


In [ ]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()

In [ ]:

del normalized_data
del OS_Scan_data_total
del feature_data



In [ ]:
# Define the autoencoder architecture with additional layers
input_dim = X_train.shape[1]
encoding_dim = 100  # Adjust the size of the encoding layer as per your requirements

input_data = keras.Input(shape=(input_dim,))
encoded = keras.layers.Dense(256, activation='relu')(input_data)
dropout_encoded = keras.layers.Dropout(0.1)(encoded)
encoded2 = keras.layers.Dense(128, activation='relu')(dropout_encoded)
dropout_encoded2 = keras.layers.Dropout(0.1)(encoded2)
encoded3 = keras.layers.Dense(64, activation='relu')(dropout_encoded2)
dropout_encoded3 = keras.layers.Dropout(0.1)(encoded3)
encoded4 = keras.layers.Dense(128, activation='relu')(dropout_encoded3)
dropout_encoded4 = keras.layers.Dropout(0.1)(encoded4)
encoded5 = keras.layers.Dense(256, activation='relu')(dropout_encoded4)
dropout_encoded5 = keras.layers.Dropout(0.1)(encoded5)
decoded = keras.layers.Dense(input_dim, activation='sigmoid')(dropout_encoded5)

autoencoder = keras.Model(input_data, decoded)

from tensorflow.keras.optimizers import Adam

# Assuming 'autoencoder' is the model you want to compile
learning_rate = 0.001  # Learning rate value

# Create an optimizer with the desired learning rate
optimizer = Adam(learning_rate=learning_rate)
autoencoder.compile(optimizer=optimizer, loss='mse')

# Train the autoencoder
autoencoder.fit(X_train, X_train, epochs=10, batch_size=5000)


# Save the model
model_path = "/content/drive/MyDrive/Colab Notebooks/xNIDS/Models/kitsune.h5"
autoencoder.save(model_path)
print("Model saved.")

# Load the model
loaded_model = keras.models.load_model(model_path)
print("Model loaded.")



# Reconstruction
reconstructed_data = autoencoder.predict(X_test)

# Compute reconstruction error
reconstruction_error = np.mean(np.square(X_test - reconstructed_data), axis=1)

# Set the threshold for anomaly detection
threshold = np.mean(reconstruction_error) + 2 * np.std(reconstruction_error)

# Classify anomalies
predicted_labels = (reconstruction_error > threshold).astype(int)

# Evaluate performance
accuracy = np.mean(predicted_labels == Y_test)

print(f"Accuracy: {accuracy}")


In [ ]:


# Reconstruction
reconstructed_data1 = autoencoder.predict(normalized_df)

# Compute reconstruction error
reconstruction_error1 = np.mean(np.square(normalized_df - reconstructed_data1), axis=1)

In [ ]:
# Create scatter plot
fig, ax = plt.subplots()
ax.scatter(OS_Scan_label_total.index, reconstruction_error1, c=label_data, cmap='coolwarm')

# Format x-axis as dates

# Set axis labels and title
ax.set_xlabel('Timestamp')
# Set y-axis to log scale with negative values
# Set y-axis to log scale
#ax.set_yscale('log')
ax.set_ylabel('Reconstruction Error')
ax.set_title('Reconstruction Error vs. Timestamp')
# Plot the horizontal line on the Axes object
ax.axhline(y=threshold, color='red', linestyle='--')
# Plot the vertical line
ax.axvline(x =1000000, color='blue', linestyle='-.')
# Show the plot
plt.show()

In [ ]:
import psutil

# Get disk usage information
disk_usage = psutil.disk_usage('/')

# Print the used disk space in bytes
print("Used disk space:", disk_usage.used)

# Print the used disk space in a human-readable format
print("Used disk space:", psutil.disk_usage('/').used / (1024**3), "GB")

In [ ]:
# Prepare data for explanations
threshold

In [ ]:
predicted_labels

In [ ]:
Y_test = label_data[split_index:]

In [ ]:
kitsune_false_positives = []  # Store indices of false positives
kitsune_false_negatives = []  # Store indices of false negatives
kitsune_true_positives = []  # Store indices of false negatives
for i in range(len(reconstruction_error)):
    if predicted_labels[i] != Y_test.values[i]:
        if predicted_labels[i] == 1:  # False positive
            kitsune_false_positives.append(i)
        else:  # False negative
            kitsune_false_negatives.append(i)
    else:
        if predicted_labels[i] == 1:  # True positive
            kitsune_true_positives.append(i)

In [ ]:
kfn = kitsune_false_negatives[39000]
kfn

In [ ]:
reconstruction_error[kfn]

In [ ]:
Y_test.iloc[kfn]

In [ ]:
label_data.iloc[1000000+kfn]

In [ ]:
kitsune_selected_fn_rows =  pd.DataFrame(X_test[kfn-10:kfn])
kitsune_selected_fn_rows.to_csv('/content/drive/MyDrive/Colab Notebooks/xNIDS/Data/kitsune_selected_fn_rows.csv', index=False)
kitsune_selected_fn_rows

In [ ]:
ktp = kitsune_true_positives[20000]
ktp

In [ ]:
reconstruction_error[ktp]

In [ ]:
Y_test.iloc[ktp]

In [ ]:
label_data.iloc[1000000+ktp]

In [ ]:
kitsune_selected_tp_rows =  pd.DataFrame(X_test[ktp-10:ktp])
kitsune_selected_tp_rows.to_csv('/content/drive/MyDrive/Colab Notebooks/xNIDS/Data/kitsune_selected_tp_rows.csv', index=False)
kitsune_selected_tp_rows

In [ ]:
len(kitsune_false_positives)

In [ ]:
kfp = kitsune_false_positives[20000]
kfp

In [ ]:
reconstruction_error[kfp]

In [ ]:
kitsune_selected_fp_rows = pd.DataFrame(X_test[kfp-10:kfp])
# Save DataFrame as CSV
kitsune_selected_fp_rows.to_csv('/content/drive/MyDrive/Colab Notebooks/xNIDS/Data/kitsune_selected_fp_rows.csv', index=False)
kitsune_selected_fp_rows